In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler


In [3]:
# Load dataset
df = pd.read_csv("credit_risk_dataset.csv")
df.head()


C:\Users\Srashti rastogi\AppData\Local\Temp\ipykernel_3652\1886423131.py:2: DtypeWarning: Columns (19,55) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("credit_risk_dataset.csv")


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
0,1077501,1296599,5000,5000,4975.0,36 months,10.65,162.87,B,B2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1077430,1314167,2500,2500,2500.0,60 months,15.27,59.83,C,C4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1077175,1313524,2400,2400,2400.0,36 months,15.96,84.33,C,C5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1076863,1277178,10000,10000,10000.0,36 months,13.49,339.31,C,C1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1075358,1311748,3000,3000,3000.0,60 months,12.69,67.79,B,B5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.columns


Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title',
       'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
       'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose',
       'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs',
       'earliest_cr_line', 'inq_last_6mths', 'mths_since_last_delinq',
       'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal',
       'revol_util', 'total_acc', 'initial_list_status', 'out_prncp',
       'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp',
       'total_rec_int', 'total_rec_late_fee', 'recoveries',
       'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
       'next_pymnt_d', 'last_credit_pull_d', 'collections_12_mths_ex_med',
       'mths_since_last_major_derog', 'policy_code', 'application_type',
       'annual_inc_joint', 'dti_joint', 'verification_status_joint',
    

In [5]:
# Select important columns
df = df[[
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "annual_inc",
    "dti",
    "loan_status"
]]

In [6]:
df.columns

Index(['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'annual_inc',
       'dti', 'loan_status'],
      dtype='object')

In [7]:
# Keep only binary target rows
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])]

# Convert target to numeric
df["loan_status"] = df["loan_status"].map({
    "Fully Paid": 0,
    "Charged Off": 1
})

In [8]:
df.head()

,loan_amnt,term,int_rate,installment,grade,annual_inc,dti,loan_status
0,5000,36 months,10.65,162.87,B,24000.0,27.65,0
1,2500,60 months,15.27,59.83,C,30000.0,1.00,1
2,2400,36 months,15.96,84.33,C,12252.0,8.72,0
3,10000,36 months,13.49,339.31,C,49200.0,20.00,0
5,5000,36 months,7.90,156.46,A,36000.0,11.20,0


In [9]:
# Clean term
df["term"] = df["term"].str.replace(" months", "")
df["term"] = df["term"].astype(int)

In [10]:
df.head()

,loan_amnt,term,int_rate,installment,grade,annual_inc,dti,loan_status
0,5000,36,10.65,162.87,B,24000.0,27.65,0
1,2500,60,15.27,59.83,C,30000.0,1.00,1
2,2400,36,15.96,84.33,C,12252.0,8.72,0
3,10000,36,13.49,339.31,C,49200.0,20.00,0
5,5000,36,7.90,156.46,A,36000.0,11.20,0


In [12]:
# Clean interest rate
if df["int_rate"].dtype == "object":
    df["int_rate"] = df["int_rate"].str.replace("%", "").astype(float)


In [13]:
# Encode grade
df = pd.get_dummies(df, columns=["grade"], drop_first=True)

In [14]:
df.columns

Index(['loan_amnt', 'term', 'int_rate', 'installment', 'annual_inc', 'dti',
       'loan_status', 'grade_B', 'grade_C', 'grade_D', 'grade_E', 'grade_F',
       'grade_G'],
      dtype='object')

In [15]:
# Split features & target
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train model
model = LogisticRegression(max_iter=3000)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.8209704516256547


In [16]:
# Save files
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(X.columns, open("columns.pkl", "wb"))

print("All files saved successfully!")

All files saved successfully!
